# 🎯 Notebook 07 — Final Test Set Evaluation

**Loads pre-computed results from retrain_final.py**

Results shown:
- Full metrics table (Accuracy, Precision, Recall, Specificity, F1, AUC, MCC)
- Confusion matrix of best model
- ROC curves for all 6 classifiers
- Per-magnification breakdown
- CV vs Test comparison

**Run each cell with Shift+Enter**

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, r'D:\Projects\ML')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (roc_curve, auc, confusion_matrix,
                              classification_report)
from src.config import (FEATURES_FUSED_CSV, FIGURES_DIR,
                        MODELS_DIR, RESULTS_DIR, CLASS_NAMES)
from src.utils import set_plot_style

set_plot_style()
print('Ready!')

## 1. Load Results

In [ ]:
# Load pre-computed results
results_df = pd.read_csv(RESULTS_DIR / 'test_results.csv')
mag_df     = pd.read_csv(RESULTS_DIR / 'per_magnification_results.csv')

print('Test Results:')
print('='*80)
print(results_df[['Model','Accuracy','Precision','Recall',
                   'Specificity','F1','ROC_AUC','MCC']].to_string(index=False))
print('='*80)

best_name = results_df.loc[results_df['F1'].idxmax(), 'Model']
best      = results_df.loc[results_df['F1'].idxmax()]
print(f'\nBest Model (by F1): {best_name}')
print(f'CV F1   (reported): {best["CV_F1"]}% (+/- {best["CV_F1_std"]}%)')
print(f'Test Accuracy     : {best["Accuracy"]}%')
print(f'Test F1           : {best["F1"]}%')

## 2. Confusion Matrix — Best Model

In [ ]:
# Load the saved figure
img = plt.imread(str(FIGURES_DIR / '07_confusion_matrix_test.png'))
fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(img)
ax.axis('off')
ax.set_title(f'Confusion Matrix — {best_name} (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

## 3. ROC Curves — All Models

In [ ]:
# Load test data and models, plot ROC
df = pd.read_csv(FEATURES_FUSED_CSV)
feat_cols = [c for c in df.columns if c.startswith('glcm_') or c.startswith('lbp_')]
test_df = df[df['split'] == 'test']
X_te = test_df[feat_cols].values.astype('float32')
y_te = test_df['label'].values

model_files = {
    'Logistic Regression': 'logistic_regression.pkl',
    'k-NN': 'knn.pkl', 'LDA': 'lda.pkl',
    'SVM': 'svm.pkl', 'Decision Tree': 'decision_tree.pkl',
    'Random Forest': 'random_forest.pkl',
}

colors = ['#7c3aed','#2563eb','#059669','#d97706','#dc2626','#db2777']
fig, ax = plt.subplots(figsize=(9, 7))

for i, (name, fname) in enumerate(model_files.items()):
    model  = joblib.load(MODELS_DIR / fname)
    y_prob = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    ax.plot(fpr, tpr, color=colors[i], lw=2,
            label=f'{name} (AUC={auc(fpr,tpr):.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Final Test Set', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '07_roc_curves_test.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. All Metrics Bar Chart

In [ ]:
metrics = ['Accuracy','Precision','Recall','F1','ROC_AUC']
x = np.arange(len(results_df))
w = 0.15
colors2 = ['#7c3aed','#2563eb','#059669','#d97706','#ef4444']

fig, ax = plt.subplots(figsize=(14, 6))
for i, m in enumerate(metrics):
    ax.bar(x + i*w, results_df[m], w, label=m, color=colors2[i], edgecolor='#3d3f4e')

ax.set_xticks(x + w*2)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 110)
ax.set_title('All Classifiers — Full Metrics (Test Set)', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '07_all_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Per-Magnification Results

In [ ]:
print(f'Per-Magnification Results — {best_name}')
print(mag_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(mag_df))
w = 0.25
ax.bar(x-w, mag_df['Accuracy'], w, label='Accuracy',  color='#7c3aed')
ax.bar(x,   mag_df['F1'],       w, label='F1',        color='#2563eb')
ax.bar(x+w, mag_df['ROC_AUC'], w, label='ROC-AUC',   color='#059669')
ax.set_xticks(x)
ax.set_xticklabels(mag_df['Magnification'])
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 110)
ax.set_title(f'Performance per Magnification — {best_name}', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '07_per_magnification.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. CV vs Test Comparison (Key Insight)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(results_df))
w = 0.35

ax.bar(x-w/2, results_df['CV_F1'], w, label='CV F1 (Patient-Aware)',
       color='#7c3aed', edgecolor='#3d3f4e',
       yerr=results_df['CV_F1_std'], capsize=5)
ax.bar(x+w/2, results_df['F1'], w, label='Test F1',
       color='#059669', edgecolor='#3d3f4e')

ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.set_ylabel('F1 Score (%)')
ax.set_ylim(0, 105)
ax.set_title('CV F1 vs Test F1 — All Classifiers', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / '07_cv_vs_test.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

In [ ]:
print('='*60)
print('  NOTEBOOK 07 COMPLETE — Final Evaluation')
print('='*60)
print(f'  Best Model       : {best_name}')
print(f'  CV F1 (primary)  : {best["CV_F1"]}% (+/- {best["CV_F1_std"]}%)')
print(f'  Test Accuracy    : {best["Accuracy"]}%')
print(f'  Test F1          : {best["F1"]}%')
print(f'  Test ROC-AUC     : {best["ROC_AUC"]}%')
print(f'  Test MCC         : {best["MCC"]}')
print()
print('  Next: Notebook 08 — Ablation Study')
print('='*60)